# Leaderboard Backtest

Reconstructs **what weekly leaderboard scores would have been** from the existing
`events` + `study_sessions` tables, using the helper functions defined in
`services.py`. Goal: validate the scoring formula against real (or near-real)
user behaviour **before** the live weekly_scores table starts accumulating.

**How to run:**
```
pip install -r requirements-dev.txt    # installs pandas, matplotlib, jupyter
jupyter notebook analysis/leaderboard_backtest.ipynb
```

**Inputs:** `studybuddy.db` in the repo root (set `DB_PATH` below).

**Outputs:** per-(user, week) component breakdown + top-10 ranked table +
component-correlation matrix. Empty/sparse DBs are handled gracefully — the
notebook just produces empty plots, not errors.

In [ ]:
import json
import os
import sqlite3
import sys
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
import pytz

# Make services helpers importable when run from repo root
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from services import (
    piecewise_time_pts,
    streak_multiplier,
    user_calendar_keys,
)

DB_PATH = os.path.join(REPO_ROOT, "studybuddy.db")
print(f"DB: {DB_PATH} (exists: {os.path.exists(DB_PATH)})")

## 1. Load raw tables

Three tables drive the reconstruction:
- `events` — append-only log of scoring-relevant actions (quiz, MCQ, task, card)
- `study_sessions` — completed Pomodoro sessions (time pts source)
- `users` — for the local timezone needed to compute week boundaries

In [ ]:
conn = sqlite3.connect(DB_PATH)

events = pd.read_sql_query(
    "SELECT id, user_id, event_name, properties, created_at FROM events ORDER BY id",
    conn,
)
sessions = pd.read_sql_query(
    "SELECT id, user_id, duration_minutes, created_at FROM study_sessions ORDER BY id",
    conn,
)
users = pd.read_sql_query(
    "SELECT user_id, timezone, current_streak FROM users",
    conn,
)
conn.close()

print(f"events: {len(events):>6}  sessions: {len(sessions):>6}  users: {len(users):>6}")
events.head()

## 2. Parse event properties + assign local week

`events.properties` is JSON. We extract the bool flags needed for scoring:
- `is_correct` for quiz/MCQ
- `succeeded` for tasks
- `is_new` + `quality` for flashcards (correct = quality ≥ 3)

We also convert `created_at` (stored as naive UTC string by `datetime('now')`)
to the **user's local datetime**, since week_iso is computed in user-local TZ.

In [ ]:
if len(events) > 0:
    events["props"] = events["properties"].apply(json.loads)
    events["is_correct"] = events["props"].apply(
        lambda p: p.get("is_correct", p.get("succeeded"))
    )
    events["is_new"] = events["props"].apply(lambda p: p.get("is_new"))
    events["quality"] = events["props"].apply(lambda p: p.get("quality"))

user_tz = users.set_index("user_id")["timezone"].to_dict()

def to_local_dt(row):
    tz_name = user_tz.get(row.user_id, "Europe/Moscow")
    try:
        tz = pytz.timezone(tz_name)
    except pytz.UnknownTimeZoneError:
        tz = pytz.timezone("Europe/Moscow")
    # created_at is stored as 'YYYY-MM-DD HH:MM:SS' in UTC
    dt_utc = datetime.strptime(row.created_at, "%Y-%m-%d %H:%M:%S").replace(
        tzinfo=pytz.UTC
    )
    return dt_utc.astimezone(tz)

for df in (events, sessions):
    if len(df) > 0:
        df["local_dt"] = df.apply(to_local_dt, axis=1)
        df["local_date"] = df["local_dt"].dt.strftime("%Y-%m-%d")
        df["week_iso"] = df["local_dt"].dt.strftime("%G-W%V")

events.head() if len(events) else "no events yet"

## 3. Reconstruct scores chronologically

Walk events + sessions in time order, maintaining per-(user, local_date) state
(daily counters mirroring the live `daily_score_counters` table). For each
scoring trigger, apply the helper functions to compute pts.

Output: `weekly_scores_df` with one row per (user, week_iso) and component breakdown.

In [ ]:
from collections import defaultdict

# Per-(user, local_date) intra-day state
daily = defaultdict(lambda: {
    "time_minutes": 0, "time_pts": 0.0,
    "task_count": 0,
    "quiz_count": 0, "quiz_series": 0,
    "cards_count": 0,
})
# Per-(user, week_iso) component totals
weekly = defaultdict(lambda: {
    "time_pts": 0.0, "task_pts": 0, "quiz_pts": 0, "card_pts": 0
})

# --- TIME PTS: walk sessions ---
if len(sessions) > 0:
    for row in sessions.itertuples():
        key_d = (row.user_id, row.local_date)
        d = daily[key_d]
        start = d["time_minutes"]
        end = start + row.duration_minutes
        pts = piecewise_time_pts(start, end)
        d["time_minutes"] = min(end, 240)
        d["time_pts"] += pts
        weekly[(row.user_id, row.week_iso)]["time_pts"] += pts

# --- TASK / QUIZ / MCQ / CARD: walk events ---
if len(events) > 0:
    for row in events.itertuples():
        key_d = (row.user_id, row.local_date)
        key_w = (row.user_id, row.week_iso)
        d = daily[key_d]
        name = row.event_name

        if name == "task_attempted" and row.is_correct is True:
            if d["task_count"] < 5:
                d["task_count"] += 1
                weekly[key_w]["task_pts"] += 40

        elif name in ("quiz_answered", "mcq_answered"):
            if row.is_correct is True:
                if d["quiz_count"] < 25:
                    d["quiz_count"] += 1
                    d["quiz_series"] += 1
                    pts = 5
                    if d["quiz_series"] % 3 == 0:
                        pts += 15
                    weekly[key_w]["quiz_pts"] += pts
            elif row.is_correct is False:
                d["quiz_series"] = 0

        elif name == "flashcard_reviewed":
            q = row.quality
            if q is not None and q >= 3 and d["cards_count"] < 8:
                d["cards_count"] += 1
                weekly[key_w]["card_pts"] += (3 if row.is_new else 5)

# Materialize to DataFrame
rows = [
    {"user_id": uid, "week_iso": wk, **comps}
    for (uid, wk), comps in weekly.items()
]
ws_df = pd.DataFrame(rows)
if len(ws_df) > 0:
    ws_df["total_base"] = (
        ws_df["time_pts"] + ws_df["task_pts"] + ws_df["quiz_pts"] + ws_df["card_pts"]
    )
    user_streak = users.set_index("user_id")["current_streak"].to_dict()
    ws_df["multiplier"] = ws_df["user_id"].map(
        lambda u: streak_multiplier(user_streak.get(u, 0))
    )
    ws_df["total_final"] = (ws_df["total_base"] * ws_df["multiplier"]).round(2)
    ws_df = ws_df.sort_values(["week_iso", "total_final"], ascending=[True, False])
print(f"reconstructed {len(ws_df)} (user, week) rows")
ws_df.head(20) if len(ws_df) else "no weekly rows reconstructed"

## 4. Top-10 per week

What the leaderboard *would have shown* each of the past few weeks.

In [ ]:
if len(ws_df) > 0:
    for wk in sorted(ws_df["week_iso"].unique())[-4:]:
        print(f"\n=== {wk} ===")
        top10 = ws_df[ws_df["week_iso"] == wk].head(10)[
            ["user_id", "time_pts", "task_pts", "quiz_pts", "card_pts",
             "total_base", "multiplier", "total_final"]
        ]
        print(top10.to_string(index=False))
else:
    print("empty reconstruction — DB has no scoring events yet")

## 5. Component breakdown (latest week)

Stacked bar chart of each top-10 user's score, broken down by component.
Visual sanity check: is one component pathologically dominating?

In [ ]:
if len(ws_df) > 0:
    latest_wk = sorted(ws_df["week_iso"].unique())[-1]
    top10 = ws_df[ws_df["week_iso"] == latest_wk].head(10).set_index("user_id")
    cols = ["time_pts", "task_pts", "quiz_pts", "card_pts"]
    ax = top10[cols].plot(
        kind="bar", stacked=True, figsize=(10, 5),
        title=f"Top-10 component breakdown — {latest_wk}",
    )
    ax.set_xlabel("user_id")
    ax.set_ylabel("weekly pts (before multiplier)")
    plt.tight_layout()
    plt.show()
else:
    print("empty — skipping chart")

## 6. Component correlations

Pearson correlations between each component and the total score, across all
(user, week) pairs. A component with correlation ≈1 dominates the total;
near-zero means it barely contributes.

Healthy formula: tasks correlate the strongest (mission lever per
LEADERBOARD.md), but time / quizzes / cards each correlate non-trivially.

In [ ]:
if len(ws_df) > 1:
    corr_cols = ["time_pts", "task_pts", "quiz_pts", "card_pts", "total_base"]
    print(ws_df[corr_cols].corr().round(3))
else:
    print("insufficient data for correlations (need ≥2 reconstructed rows)")

## Notes for extension

- **Pre-Phase-0 events** lack the `is_new` flag on flashcard_reviewed. For
  those, you can infer it: first `flashcard_reviewed` event for a given
  `(user_id, card_hash)` = new. Add as a pandas group-by transform if needed.
- **Streak history.** Currently the multiplier uses `users.current_streak`
  (which is the streak *now*, not at week's end). For a true historical
  multiplier per-week, would need to walk session dates and derive streak.
  Out of scope for v1 of the backtest.
- **Segments.** Newbie segment (`< 7 days since registration`) and main
  segment aren't split in this notebook — would join `users.created_at`
  vs. each week's start. Easy add when needed.
- **Plots.** Add per-week trend lines for the top-3 users to show stability
  of rankings week-over-week.